In [ ]:

py_path <- Sys.which("python")

if (py_path == "") {
  py_path <- Sys.which("python3")
}

Sys.setenv(RETICULATE_PYTHON = py_path)

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(reticulate)
  library(CellEnergy)
})

cat("[INFO] Python used by reticulate:\n")
print(reticulate::py_config())

cat("[INFO] Python module check:\n")
cat("pandas:", reticulate::py_module_available("pandas"), "\n")
cat("numpy :", reticulate::py_module_available("numpy"), "\n")
cat("scipy :", reticulate::py_module_available("scipy"), "\n\n")



meta_path <- "path/GSE138536/GSE138536_HBC_metadata.txt/GSE138536_HBC_metadata.txt"

expr_path <- "path/GSE138536/GSE138536_HBC_TranscriptMatrixSalmon_TPM.txt/GSE138536_HBC_TranscriptMatrixSalmon_TPM.txt"

out_dir <- "path/GSE138536/Seurat_CellEnergy_full_pipeline"

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

label_col <- "Seurat_cluster"
n_hvg <- 2000
min_cells <- 3
min_features <- 200

if (!file.exists(meta_path)) {
  stop("[ERROR] Metadata file not found: ", meta_path)
}

if (!file.exists(expr_path)) {
  stop("[ERROR] Expression file not found: ", expr_path)
}



write_matrix_with_id <- function(mat, file, id_col = "ID") {
  mat <- as.matrix(mat)
  df <- as.data.frame(mat, check.names = FALSE)
  df <- cbind(setNames(data.frame(rownames(df), check.names = FALSE), id_col), df)
  data.table::fwrite(df, file = file)
}

minmax_scale_columns <- function(mat) {
  mat <- as.matrix(mat)

  col_min <- apply(mat, 2, min, na.rm = TRUE)
  col_max <- apply(mat, 2, max, na.rm = TRUE)
  col_range <- col_max - col_min

  col_range[col_range == 0 | is.na(col_range)] <- 1

  scaled <- sweep(mat, 2, col_min, FUN = "-")
  scaled <- sweep(scaled, 2, col_range, FUN = "/")
  scaled[is.na(scaled)] <- 0

  return(scaled)
}

get_assay_data_compat <- function(obj, assay = "RNA", layer_name = "data", slot_name = "data") {
  DefaultAssay(obj) <- assay

  tryCatch(
    GetAssayData(obj, assay = assay, layer = layer_name),
    error = function(e) {
      GetAssayData(obj, assay = assay, slot = slot_name)
    }
  )
}



cat("[INFO] Reading metadata...\n")

metadata <- data.table::fread(
  meta_path,
  sep = "\t",
  data.table = FALSE,
  na.strings = c("NA", "NaN", "", " ")
)

cat("[INFO] Metadata dimension:\n")
print(dim(metadata))

cat("[INFO] Metadata columns:\n")
print(colnames(metadata))

if (!"UniqueID" %in% colnames(metadata)) {
  stop("[ERROR] Metadata must contain column: UniqueID")
}

if (!label_col %in% colnames(metadata)) {
  stop("[ERROR] label_col not found: ", label_col)
}

metadata$UniqueID <- as.character(metadata$UniqueID)
metadata[[label_col]] <- as.character(metadata[[label_col]])

rownames(metadata) <- metadata$UniqueID

cat("[INFO] Label distribution before alignment:\n")
print(table(metadata[[label_col]], useNA = "ifany"))



cat("[INFO] Reading TPM expression matrix...\n")

expr_df <- data.table::fread(
  expr_path,
  sep = "\t",
  data.table = FALSE,
  check.names = FALSE
)

cat("[INFO] Raw expression table dimension:\n")
print(dim(expr_df))

cat("[INFO] First 5 columns:\n")
print(head(colnames(expr_df), 5))


gene_col <- colnames(expr_df)[1]
genes <- as.character(expr_df[[gene_col]])

expr_df[[gene_col]] <- NULL

expr_mat <- as.matrix(expr_df)

suppressWarnings({
  mode(expr_mat) <- "numeric"
})

expr_mat[is.na(expr_mat)] <- 0

rownames(expr_mat) <- make.unique(genes)

cat("[INFO] Expression matrix after reading, genes x cells assumed:\n")
print(dim(expr_mat))



common_cols <- intersect(colnames(expr_mat), metadata$UniqueID)
common_rows <- intersect(rownames(expr_mat), metadata$UniqueID)

cat("[INFO] Common cells with expression columns:\n")
print(length(common_cols))

cat("[INFO] Common cells with expression rows:\n")
print(length(common_rows))

if (length(common_rows) > length(common_cols)) {
  cat("[INFO] Expression appears to be cells x genes. Transposing...\n")
  expr_mat <- t(expr_mat)
  common_cells <- intersect(colnames(expr_mat), metadata$UniqueID)
} else {
  cat("[INFO] Expression appears to be genes x cells.\n")
  common_cells <- common_cols
}

if (length(common_cells) == 0) {
  stop("[ERROR] No common cells between metadata UniqueID and expression matrix.")
}

expr_mat <- expr_mat[, common_cells, drop = FALSE]
metadata <- metadata[common_cells, , drop = FALSE]

cat("[INFO] Aligned expression matrix, genes x cells:\n")
print(dim(expr_mat))

cat("[INFO] Aligned metadata:\n")
print(dim(metadata))

cat("[INFO] Label distribution after alignment:\n")
print(table(metadata[[label_col]], useNA = "ifany"))


keep_cells <- !is.na(metadata[[label_col]]) & metadata[[label_col]] != ""

expr_mat <- expr_mat[, keep_cells, drop = FALSE]
metadata <- metadata[keep_cells, , drop = FALSE]

cat("[INFO] After removing NA labels:\n")
print(dim(expr_mat))
print(table(metadata[[label_col]], useNA = "ifany"))


gene_sum <- rowSums(expr_mat)
expr_mat <- expr_mat[gene_sum > 0, , drop = FALSE]


if (any(duplicated(rownames(expr_mat)))) {
  cat("[WARN] Duplicated gene names found. Aggregating by mean.\n")
  expr_mat <- rowsum(expr_mat, group = rownames(expr_mat), reorder = FALSE) /
    as.vector(table(rownames(expr_mat)))
}



log_tpm <- log1p(expr_mat)


seurat_obj <- CreateSeuratObject(
  counts = log_tpm,
  project = "GSE138536",
  meta.data = metadata,
  min.cells = min_cells,
  min.features = min_features
)

cat("[INFO] Seurat object:\n")
print(seurat_obj)


DefaultAssay(seurat_obj) <- "RNA"



DefaultAssay(seurat_obj) <- "RNA"

counts_layer <- GetAssayData(
  object = seurat_obj,
  assay = "RNA",
  layer = "counts"
)

seurat_obj <- SetAssayData(
  object = seurat_obj,
  assay = "RNA",
  layer = "data",
  new.data = counts_layer
)



seurat_obj <- FindVariableFeatures(
  seurat_obj,
  selection.method = "vst",
  nfeatures = min(n_hvg, nrow(seurat_obj))
)

hvg_genes <- VariableFeatures(seurat_obj)

cat("[INFO] Number of HVGs:\n")
print(length(hvg_genes))

data.table::fwrite(
  data.frame(HVG = hvg_genes),
  file.path(out_dir, paste0("HVG_", length(hvg_genes), "_gene_list.csv"))
)

seurat_obj <- ScaleData(
  seurat_obj,
  features = hvg_genes
)



metadata_after_qc <- seurat_obj@meta.data

cell_labels_after_qc <- data.frame(
  Cell = rownames(metadata_after_qc),
  Label = metadata_after_qc[[label_col]],
  tumor_vs_normal = metadata_after_qc$tumor_vs_normal,
  Clinical_subtype = metadata_after_qc$Clinical_subtype,
  Patient = metadata_after_qc$Patient,
  stringsAsFactors = FALSE
)

data.table::fwrite(
  cell_labels_after_qc,
  file.path(out_dir, "cell_labels_after_QC.csv")
)

data.table::fwrite(
  cbind(Cell = rownames(metadata_after_qc), metadata_after_qc),
  file.path(out_dir, "metadata_after_QC.csv")
)

cat("[INFO] Label distribution after QC:\n")
print(table(cell_labels_after_qc$Label, useNA = "ifany"))


log_tpm_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "data",
  slot_name = "data"
)[hvg_genes, ]

scaled_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "scale.data",
  slot_name = "scale.data"
)[hvg_genes, ]

log_tpm_hvg_cell_by_gene <- t(as.matrix(log_tpm_hvg_gene_by_cell))
log_tpm_hvg_minmax <- minmax_scale_columns(log_tpm_hvg_cell_by_gene)

write_matrix_with_id(
  log_tpm_hvg_cell_by_gene,
  file.path(out_dir, "final_logTPM_HVG_expr_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  log_tpm_hvg_minmax,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] View 1 saved:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n\n")



scaled_for_cellenergy_path <- file.path(
  out_dir,
  "scaled_HVG_expr_gene_by_cell_for_CellEnergy.csv"
)

write.csv(
  as.data.frame(as.matrix(scaled_hvg_gene_by_cell), check.names = FALSE),
  file = scaled_for_cellenergy_path,
  quote = FALSE,
  row.names = TRUE
)

cat("[INFO] Running CellEnergy::calcGEn...\n")

result <- CellEnergy::calcGEn(
  scaled_for_cellenergy_path,
  verbose = TRUE
)

GLNE <- result$GLNE
GLNE_mat <- as.matrix(GLNE)

write_matrix_with_id(
  GLNE_mat,
  file.path(out_dir, "GLNE_raw_output.csv"),
  id_col = "ID"
)

glne_rows_are_cells <- sum(rownames(GLNE_mat) %in% rownames(log_tpm_hvg_cell_by_gene))
glne_cols_are_cells <- sum(colnames(GLNE_mat) %in% rownames(log_tpm_hvg_cell_by_gene))

if (glne_rows_are_cells > glne_cols_are_cells) {
  cat("[INFO] GLNE appears to be cells x genes.\n")
  GLNE_cell_by_gene <- GLNE_mat
} else {
  cat("[INFO] GLNE appears to be genes x cells. Transposing.\n")
  GLNE_cell_by_gene <- t(GLNE_mat)
}

common_cells <- intersect(rownames(log_tpm_hvg_cell_by_gene), rownames(GLNE_cell_by_gene))
common_genes <- intersect(colnames(log_tpm_hvg_cell_by_gene), colnames(GLNE_cell_by_gene))

cat("[INFO] Common cells between View1 and GLNE:\n")
print(length(common_cells))

cat("[INFO] Common genes between View1 and GLNE:\n")
print(length(common_genes))

if (length(common_cells) == 0) {
  stop("[ERROR] No common cells between View1 and GLNE.")
}

if (length(common_genes) == 0) {
  stop("[ERROR] No common genes between View1 and GLNE.")
}

view1_final <- log_tpm_hvg_cell_by_gene[common_cells, common_genes, drop = FALSE]
GLNE_cell_by_gene <- GLNE_cell_by_gene[common_cells, common_genes, drop = FALSE]

view1_minmax_final <- minmax_scale_columns(view1_final)
GLNE_minmax <- minmax_scale_columns(GLNE_cell_by_gene)

write_matrix_with_id(
  view1_final,
  file.path(out_dir, "final_logTPM_HVG_expr_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  view1_minmax_final,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_cell_by_gene,
  file.path(out_dir, "final_GLNE_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_minmax,
  file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] View 2 saved:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n\n")



seurat_rds <- file.path(
  out_dir,
  "GSE138536_Seurat_CellEnergy_processed_TPM_no_regression.rds"
)

saveRDS(seurat_obj, file = seurat_rds)

cat("\n[DONE] GSE138536 preprocessing finished.\n")
cat("1. Labels:\n")
cat(file.path(out_dir, "cell_labels_after_QC.csv"), "\n")
cat("2. View 1:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n")
cat("3. View 2:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n")
cat("4. Seurat RDS:\n")
cat(seurat_rds, "\n")

[INFO] Python used by reticulate:
python:         /home/zhanghang/Anaconda/envs/r_cellenergy/bin/python
libpython:      /home/zhanghang/Anaconda/envs/r_cellenergy/lib/libpython3.14.so
pythonhome:     /home/zhanghang/Anaconda/envs/r_cellenergy:/home/zhanghang/Anaconda/envs/r_cellenergy
version:        3.14.5 | packaged by conda-forge | (main, May 20 2026, 00:15:36) [GCC 14.3.0]
numpy:          /home/zhanghang/Anaconda/envs/r_cellenergy/lib/python3.14/site-packages/numpy
numpy_version:  2.4.6

NOTE: Python version was forced by RETICULATE_PYTHON
[INFO] Python module check:
pandas: TRUE 
numpy : TRUE 
scipy : TRUE 

[INFO] Reading metadata...


Warning message in data.table::fread(meta_path, sep = "\t", data.table = FALSE, :
“na.strings[4]==" " consists only of whitespace, ignoring. strip.white==TRUE (default) and "" is present in na.strings, so any number of spaces in string columns will already be read as <NA>.”


[INFO] Metadata dimension:
[1] 1902   14
[INFO] Metadata columns:
 [1] "UniqueID"         "Patient"          "tumor_vs_normal"  "Clinical_subtype"
 [5] "Seurat_cluster"   "ER_status"        "PR_status"        "HER_status"      
 [9] "PC1"              "PC2"              "PC3"              "CytoTRACE_basal" 
[13] "CytoTRACE_LP"     "CytoTRACE_ML"    
[INFO] Label distribution before alignment:

             Basal Luminal progenitor     Mature luminal 
               660                532                710 
[INFO] Reading TPM expression matrix...


Warning message in data.table::fread(expr_path, sep = "\t", data.table = FALSE, :
“Detected 1902 column names but the data has 1903 columns (i.e. invalid file). Added an extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”


[INFO] Raw expression table dimension:
[1] 56269  1903
[INFO] First 5 columns:
[1] "V1"            "A10_004_SU2_N" "A1_004_SU2_N"  "A1_005_SU2_T" 
[5] "A11_004_SU2_N"
[INFO] Expression matrix after reading, genes x cells assumed:
[1] 56269  1902
[INFO] Common cells with expression columns:
[1] 1902
[INFO] Common cells with expression rows:
[1] 0
[INFO] Expression appears to be genes x cells.
[INFO] Aligned expression matrix, genes x cells:
[1] 56269  1902
[INFO] Aligned metadata:
[1] 1902   14
[INFO] Label distribution after alignment:

             Basal Luminal progenitor     Mature luminal 
               660                532                710 
[INFO] After removing NA labels:
[1] 56269  1902

             Basal Luminal progenitor     Mature luminal 
               660                532                710 


...

[INFO] Number of HVGs:
[1] 2000


Centering and scaling data matrix



[INFO] Label distribution after QC:

             Basal Luminal progenitor     Mature luminal 
               660                532                710 
[OK] View 1 saved:
/home/zhanghang/GSE138536/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 

[INFO] Running CellEnergy::calcGEn...


2026-06-11 23:09:00.085049: Starting calculation.

2026-06-11 23:09:21.872603: Complete GLNE and cell Energy calculation.



[INFO] GLNE appears to be genes x cells. Transposing.
[INFO] Common cells between View1 and GLNE:
[1] 1902
[INFO] Common genes between View1 and GLNE:
[1] 2000
[OK] View 2 saved:
/home/zhanghang/GSE138536/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 


[DONE] GSE138536 preprocessing finished.
1. Labels:
/home/zhanghang/GSE138536/Seurat_CellEnergy_full_pipeline/cell_labels_after_QC.csv 
2. View 1:
/home/zhanghang/GSE138536/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 
3. View 2:
/home/zhanghang/GSE138536/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 
4. Seurat RDS:
/home/zhanghang/GSE138536/Seurat_CellEnergy_full_pipeline/GSE138536_Seurat_CellEnergy_processed_TPM_no_regression.rds 
